# phase 2 — layer-0 interventions that flatten the first answer token, across prompts
See `PLAN.md`. `H1` as phase 11: entropy (bits) of the first answer-position distribution, Qwen3-8B bf16, thinking off.
Every arm perturbs the residual stream after block 0 by `eps * ||h_t|| * d_t` with `||d_t|| = 1`, so eps is matched across arms.

In [2]:
# === CELL 1 — rig: model, scaffold, prompts, clean H1 reproduces phase 11 ===========================
import torch, torch.nn.functional as F, math, json, time, inspect, unicodedata, os
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
LN2 = math.log(2); Q = "what shall i do today"
MODEL_ID = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0"); model.eval(); model.requires_grad_(False)
dev, EOS, V = model.device, tokenizer.eos_token_id, model.get_input_embeddings().weight.shape[0]
CEIL = math.log2(V)
_LTK = "logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters else "num_logits_to_keep"
def _chat(t):
    try: return tokenizer.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return tokenizer.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True)
def scaffold(query):
    """ids of the full chat prompt, plus (start, end) token span of the user's query text."""
    s = _chat(query); enc = tokenizer(s, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc["input_ids"], enc["offset_mapping"]; i0 = s.index(query); i1 = i0 + len(query)
    j0 = next(t for t,(a,b) in enumerate(offs) if b > i0); j1 = next(t for t,(a,b) in enumerate(offs) if a >= i1)
    return ids, (j0, j1)
def entropy_bits(lg):
    lp = F.log_softmax(lg.float(), -1); return -(lp.exp()*lp).sum(-1)/LN2
@torch.no_grad()
def H1_single(ids):
    return float(entropy_bits(model(torch.tensor([ids], device=dev), **{_LTK:1}).logits[0,-1]))
PROMPTS = json.loads('{"train": ["How do I get better at public speaking?", "Write a short poem about autumn leaves.", "What\'s the difference between a virus and a bacterium?", "Can you help me plan a two-day trip to Lisbon?", "Write a Python function that checks whether a string is a palindrome.", "How do I politely decline a meeting invitation?", "What causes the seasons on Earth?", "Give me five ideas for a birthday present for my dad.", "How does compound interest work?", "I can\'t focus when I work from home. Any advice?", "Explain what a hash map is to a beginner.", "Draft a friendly reminder email about an overdue invoice.", "What should I know before adopting a cat?", "Summarise the plot of Romeo and Juliet in three sentences.", "How do I fix a leaking tap?", "What\'s a good stretching routine for someone who sits all day?", "Why is the sky blue?", "Write a haiku about coffee.", "How do I convert Celsius to Fahrenheit?", "What\'s the best way to learn a new language as an adult?", "Recommend three podcasts about history.", "How do I make my sourdough starter more active?", "Explain the rules of chess briefly.", "What are some good questions to ask at the end of a job interview?"], "held_out": ["what shall i do today", "What shall I do today?", "What\'s a quick and healthy dinner I can make tonight?", "Give me a 3-sentence pep talk for a Monday morning.", "Explain how a rainbow forms, simply.", "Suggest a good book to read this weekend and why.", "What are three tips to sleep better?", "How do I write a good cover letter?", "What is machine learning, in one paragraph?", "Give me a simple recipe for pancakes.", "How can I save more money each month?", "Write a SQL query that returns the ten most recent orders.", "What\'s the capital of Australia, and why isn\'t it Sydney?", "Help me name my new golden retriever puppy.", "How long should I boil an egg for a runny yolk?", "Tell me a fun fact about octopuses."]}')
TRAIN, HELD = PROMPTS["train"], PROMPTS["held_out"]; ALL = TRAIN + HELD
ids_Q, span_Q = scaffold(Q); h = H1_single(ids_Q)
print(f"transformers {__import__('transformers').__version__} torch {torch.__version__} | V {V} ceiling {CEIL:.3f}")
print(f"clean H1({Q!r}) = {h:.4f} (phase 11: 0.2366) | {len(ids_Q)} tokens, query span {span_Q}")
assert abs(h-0.2366) < 0.01, "rig does not reproduce phase 11 — STOP"
print(f"prompts: {len(TRAIN)} train / {len(HELD)} held-out")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

transformers 5.16.1 torch 2.11.0+cu128 | V 151936 ceiling 17.213
clean H1('what shall i do today') = 0.2366 (phase 11: 0.2366) | 17 tokens, query span (3, 8)
prompts: 24 train / 16 held-out


In [3]:
# === CELL 2 — left-padded batch of all 40 prompts, position ids, and the three position masks =======
SC = [scaffold(p) for p in ALL]
L = max(len(i) for i,_ in SC); N = len(ALL)
IDS = torch.full((N, L), EOS, dtype=torch.long); ATT = torch.zeros((N, L), dtype=torch.long)
M_ALL = torch.zeros((N, L), dtype=torch.bool); M_USER = torch.zeros((N, L), dtype=torch.bool); M_LAST = torch.zeros((N, L), dtype=torch.bool)
for r,(ids,(j0,j1)) in enumerate(SC):
    off = L - len(ids); IDS[r, off:] = torch.tensor(ids); ATT[r, off:] = 1
    M_ALL[r, off:] = True; M_USER[r, off+j0:off+j1] = True; M_LAST[r, L-1] = True
IDS, ATT = IDS.to(dev), ATT.to(dev)
POS = (ATT.cumsum(-1) - 1).clamp(min=0)
MASKS = {"all": M_ALL.to(dev), "user": M_USER.to(dev), "last": M_LAST.to(dev)}
TR = torch.arange(N, device=dev) < len(TRAIN); HO = ~TR
def H1_batch(rows=None):
    """H1 for every prompt (or a row subset) under whatever intervention is currently set. Differentiable."""
    sl = slice(None) if rows is None else rows
    lg = model(input_ids=IDS[sl], attention_mask=ATT[sl], position_ids=POS[sl], **{_LTK:1}).logits[:, -1]
    return entropy_bits(lg)
with torch.no_grad(): H_clean = H1_batch()
iq = ALL.index(Q)
print(f"batched clean H1({Q!r}) = {H_clean[iq]:.4f} vs single {h:.4f} (bf16 batch-shape drift, phase 11 saw ~0.02-0.04)")
print(f"clean: train mean {H_clean[TR].mean():.3f} sd {H_clean[TR].std():.3f} | held-out mean {H_clean[HO].mean():.3f} sd {H_clean[HO].std():.3f} | max {H_clean.max():.3f} ({ALL[int(H_clean.argmax())]!r})")
print("query-token counts:", [int(m) for m in M_USER.sum(1)][:8], "... | seq len", L)


batched clean H1('what shall i do today') = 0.2331 vs single 0.2366 (bf16 batch-shape drift, phase 11 saw ~0.02-0.04)
clean: train mean 0.621 sd 0.616 | held-out mean 0.167 sd 0.322 | max 1.579 ('Give me five ideas for a birthday present for my dad.')
query-token counts: [9, 8, 12, 12, 13, 9, 7, 12] ... | seq len 27


In [4]:
# === CELL 3 — the layer-0 hook: const / lin8 / rand directions, scale, and the two ablations ==========
INT = dict(kind=None, u=None, U=None, Vm=None, eps=0.0, mask=None, scale=1.0, prefill_only=True, live_mask=None)
def _unit(x): return x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
def _layer0_hook(mod, inp, out):
    if INT["kind"] is None: return out
    hs = out[0] if isinstance(out, tuple) else out
    B, T, D = hs.shape
    if INT["prefill_only"] and T == 1: return out           # decode steps untouched
    m = INT["live_mask"] if INT["live_mask"] is not None else INT["mask"]
    m = m[:B, :T].to(hs.dtype)[..., None]
    hf = hs.float(); nrm = hf.norm(dim=-1, keepdim=True)
    if INT["kind"] == "scale":
        new = hf * (1 + (INT["scale"] - 1) * m.float())
    else:
        if INT["kind"] in ("const", "rand"): d = _unit(INT["u"].float())[None, None, :].expand(B, T, D)
        elif INT["kind"] == "lin8": d = _unit((_unit(hf) @ INT["Vm"].float()) @ INT["U"].float().T)   # [B,T,8] -> [B,T,D]
        new = hf + INT["eps"] * nrm * d * m.float()
    new = new.to(hs.dtype)
    return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
H0 = model.model.layers[0].register_forward_hook(_layer0_hook)
ABL = dict(attn=False, mlp=False)
def _zero_hook(which):
    def f(mod, inp, out):
        if not ABL[which]: return out
        if isinstance(out, tuple): return (torch.zeros_like(out[0]),) + tuple(out[1:])
        return torch.zeros_like(out)
    return f
HA = model.model.layers[0].self_attn.register_forward_hook(_zero_hook("attn"))
HM = model.model.layers[0].mlp.register_forward_hook(_zero_hook("mlp"))
def set_int(kind=None, eps=0.0, mask="all", u=None, U=None, Vm=None, scale=1.0):
    INT.update(kind=kind, eps=eps, mask=MASKS[mask] if isinstance(mask, str) else mask, u=u, U=U, Vm=Vm, scale=scale, live_mask=None)
def off(): set_int(None); ABL.update(attn=False, mlp=False)
with torch.no_grad():
    off(); a = H1_batch()[iq].item()
    set_int("const", eps=0.0, u=torch.randn(4096, device=dev)); b = H1_batch()[iq].item(); off()
print(f"hook installed; eps=0 leaves H1 unchanged: {a:.4f} {b:.4f}")
D = model.config.hidden_size; print("hidden size", D)


hook installed; eps=0 leaves H1 unchanged: 0.2331 0.2331
hidden size 4096


In [5]:
# === CELL 4 — parameter-free arms: scale, ablations, random directions at matched eps ===============
EPS = [0.05, 0.1, 0.2, 0.4, 0.8]
RES = {"meta": dict(model=MODEL_ID, ceiling=CEIL, V=V, prompts=PROMPTS, eps=EPS, transformers=__import__('transformers').__version__, torch=torch.__version__),
       "clean": H_clean.tolist(), "arms": {}}
def stats(H):
    return dict(train_mean=float(H[TR].mean()), train_sd=float(H[TR].std()), train_min=float(H[TR].min()),
                held_mean=float(H[HO].mean()), held_sd=float(H[HO].std()), held_min=float(H[HO].min()), per_prompt=H.tolist())
def rec(name, H, **kw):
    RES["arms"][name] = dict(stats(H), **kw)
    s = RES["arms"][name]; print(f"{name:28s} train {s['train_mean']:6.3f}±{s['train_sd']:5.3f} (min {s['train_min']:6.3f}) | held {s['held_mean']:6.3f}±{s['held_sd']:5.3f} (min {s['held_min']:6.3f})")
with torch.no_grad():
    off(); rec("clean", H_clean)
    for s in [0.0, 0.5, 2.0, 4.0]:
        for mk in ["all", "user", "last"]:
            set_int("scale", mask=mk, scale=s); rec(f"scale{s}·{mk}", H1_batch(), kind="scale", scale=s, mask=mk)
    off()
    for w in ["attn", "mlp"]:
        ABL[w] = True; rec(f"zero_{w}0", H1_batch(), kind="ablate", which=w); ABL[w] = False
    g = torch.Generator(device="cpu").manual_seed(0)
    RANDU = [torch.randn(D, generator=g).to(dev) for _ in range(8)]
    for mk in ["all", "user", "last"]:
        for e in EPS:
            Hs = []
            for u in RANDU:
                set_int("rand", eps=e, mask=mk, u=u); Hs.append(H1_batch())
            Hs = torch.stack(Hs)                       # [8, N]
            rec(f"rand·{mk}·eps{e}", Hs.mean(0), kind="rand", eps=e, mask=mk, n_draws=8, draw_train_means=Hs[:, TR].mean(1).tolist(), draw_max=float(Hs.max()))
    off()


clean                        train  0.621±0.616 (min  0.000) | held  0.167±0.322 (min  0.000)
scale0.0·all                 train 17.213±0.000 (min 17.213) | held 17.213±0.000 (min 17.213)
scale0.0·user                train  1.419±0.218 (min  0.938) | held  1.394±0.197 (min  1.010)
scale0.0·last                train  4.123±3.421 (min  0.022) | held  4.099±3.445 (min  0.022)
scale0.5·all                 train  0.734±0.731 (min  0.002) | held  0.593±0.667 (min  0.000)
scale0.5·user                train  0.599±0.594 (min  0.000) | held  0.245±0.393 (min  0.000)
scale0.5·last                train  0.606±0.567 (min  0.000) | held  0.322±0.455 (min  0.000)
scale2.0·all                 train  0.746±0.734 (min  0.003) | held  0.385±0.509 (min  0.000)
scale2.0·user                train  0.614±0.589 (min  0.000) | held  0.234±0.345 (min  0.000)
scale2.0·last                train  0.745±0.697 (min  0.000) | held  0.304±0.453 (min  0.000)
scale4.0·all                 train  1.009±0.563 (min  0.087)

In [6]:
# === CELL 5 — fit const and lin8 directions on the 24 train prompts: objective mean(H1) - std(H1) ====
LAM, STEPS, LR = 1.0, 150, 0.05
def fit(kind, eps, mk, steps=STEPS, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    if kind == "const":
        params = {"u": torch.randn(D, generator=g).to(dev).requires_grad_(True)}
    else:
        params = {"U": (torch.randn(D, 8, generator=g) / math.sqrt(8)).to(dev).requires_grad_(True),
                  "Vm": (torch.randn(D, 8, generator=g) / math.sqrt(D)).to(dev).requires_grad_(True)}
    opt = torch.optim.Adam(params.values(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    best = (-1e9, None); hist = []
    for it in range(steps):
        set_int(kind, eps=eps, mask=mk, **params)
        with torch.enable_grad():
            H = H1_batch(TR); obj = H.mean() - LAM * H.std()
            opt.zero_grad(); (-obj).backward(); opt.step(); sched.step()
        o = float(obj); hist.append(o)
        if o > best[0]: best = (o, {k: v.detach().clone() for k, v in params.items()})
    off(); return best[1], hist
FIT = {}
t0 = time.time()
for kind in ["const", "lin8"]:
    for mk in ["all", "user", "last"]:
        for e in EPS:
            name = f"{kind}·{mk}·eps{e}"
            p, hist = fit(kind, e, mk)
            FIT[name] = p
            with torch.no_grad(): set_int(kind, eps=e, mask=mk, **p); H = H1_batch(); off()
            rec(name, H, kind=kind, eps=e, mask=mk, objective_curve=hist[::10], objective_best=max(hist))
            print(f"    {time.time()-t0:6.0f}s  obj curve: " + " ".join(f"{x:.2f}" for x in hist[::30]))
np.savez("/content/phase2_directions.npz", **{f"{n}__{k}": v.cpu().float().numpy() for n, p in FIT.items() for k, v in p.items()})
json.dump(RES, open("/content/phase2_results.json", "w"))
print("saved")


/tmp/ipykernel_2106/2377752747.py:18: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  o = float(obj); hist.append(o)


const·all·eps0.05            train  0.890±0.593 (min  0.000) | held  0.175±0.296 (min  0.000)
        29s  obj curve: 0.01 0.20 0.26 0.28 0.29
const·all·eps0.1             train  1.197±0.588 (min  0.004) | held  0.263±0.345 (min  0.000)
        58s  obj curve: 0.01 0.35 0.48 0.59 0.61
const·all·eps0.2             train  4.081±1.170 (min  1.027) | held  3.034±1.590 (min  0.001)
        86s  obj curve: 0.00 0.53 1.11 2.10 2.69
const·all·eps0.4             train 14.652±0.451 (min 12.972) | held 13.369±1.968 (min  7.869)
       115s  obj curve: 0.01 1.85 3.49 11.05 13.69
const·all·eps0.8             train 16.343±0.028 (min 16.287) | held 16.234±0.130 (min 15.821)
       144s  obj curve: 0.04 13.18 15.33 16.05 16.29
const·user·eps0.05           train  0.720±0.577 (min  0.000) | held  0.164±0.322 (min  0.000)
       172s  obj curve: 0.02 0.10 0.13 0.13 0.14
const·user·eps0.1            train  0.850±0.582 (min  0.000) | held  0.149±0.296 (min  0.000)
       201s  obj curve: 0.01 0.19 0.26 0.2

In [7]:
# === CELL 5b — the surprise: block-0 MLP ablation. Dose-response and position masks, parameter-free ====
# zero_mlp0 gave ~11.5 bits on train AND held-out with nothing fit. Replace the zero hooks with a masked scaler.
HA.remove(); HM.remove()
SUB = dict(attn=1.0, mlp=1.0, mask=None, prefill_only=True)
def _sub_hook(which):
    def f(mod, inp, out):
        s = SUB[which]
        if s == 1.0: return out
        hs = out[0] if isinstance(out, tuple) else out
        B, T, _ = hs.shape
        if SUB["prefill_only"] and T == 1: return out
        m = SUB["mask"]
        m = torch.ones((B, T), device=hs.device) if m is None else m[:B, :T].to(hs.dtype)
        new = hs * (1 + (s - 1) * m[..., None]).to(hs.dtype)
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
    return f
HA = model.model.layers[0].self_attn.register_forward_hook(_sub_hook("attn"))
HM = model.model.layers[0].mlp.register_forward_hook(_sub_hook("mlp"))
def set_sub(which=None, s=1.0, mask="all"):
    SUB.update(attn=1.0, mlp=1.0, mask=None)
    if which: SUB.update({which: s, "mask": MASKS[mask] if isinstance(mask, str) else mask})
def off():
    set_int(None); set_sub(None)
with torch.no_grad():
    off(); a = H1_batch()[iq].item(); set_sub("mlp", 0.0, "all"); b = H1_batch()[iq].item(); off()
print(f"re-hooked: clean {a:.4f} | mlp0 zeroed {b:.4f} (cell 4 zero_mlp0 on this prompt: {RES['arms']['zero_mlp0']['per_prompt'][iq]:.4f})")
with torch.no_grad():
    for which in ["mlp", "attn"]:
        for s in [0.0, 0.25, 0.5, 0.75, 1.5, 2.0, 4.0]:
            for mk in ["all", "user", "last"]:
                set_sub(which, s, mk); rec(f"{which}0×{s}·{mk}", H1_batch(), kind="sub", which=which, scale=s, mask=mk)
    off()
json.dump(RES, open("/content/phase2_results.json", "w")); print("saved")


re-hooked: clean 0.2331 | mlp0 zeroed 8.4669 (cell 4 zero_mlp0 on this prompt: 8.4669)
mlp0×0.0·all                 train 11.689±2.017 (min  5.606) | held 11.422±1.599 (min  8.467)
mlp0×0.0·user                train  0.686±0.640 (min  0.015) | held  0.736±0.762 (min  0.000)
mlp0×0.0·last                train  0.685±0.639 (min  0.001) | held  0.230±0.394 (min  0.000)
mlp0×0.25·all                train  0.602±0.578 (min  0.001) | held  0.240±0.386 (min  0.000)
mlp0×0.25·user               train  0.575±0.546 (min  0.000) | held  0.246±0.393 (min  0.000)
mlp0×0.25·last               train  0.645±0.623 (min  0.000) | held  0.196±0.384 (min  0.000)
mlp0×0.5·all                 train  0.605±0.558 (min  0.000) | held  0.218±0.393 (min  0.000)
mlp0×0.5·user                train  0.602±0.598 (min  0.000) | held  0.169±0.338 (min  0.000)
mlp0×0.5·last                train  0.641±0.623 (min  0.000) | held  0.186±0.367 (min  0.000)
mlp0×0.75·all                train  0.642±0.625 (min  0.000) | held

In [8]:
# === CELL 6 — summary table + mass-support quantiles on held-out for the headline arms ===============
import itertools
print(f"{'arm':30s} {'train':>14s} {'held':>14s} {'held-train':>10s}")
for name, s in RES["arms"].items():
    if s.get("kind") in ("const", "lin8", "rand"):
        print(f"{name:30s} {s['train_mean']:6.2f}±{s['train_sd']:4.2f}  {s['held_mean']:6.2f}±{s['held_sd']:4.2f}  {s['held_mean']-s['train_mean']:+8.2f}")
def support(kind, e, mk, p=None, rows=None):
    """per-prompt n50/n90/n99 and top-5 tokens, under the arm."""
    out = []
    with torch.no_grad():
        if kind == "clean": off()
        else: set_int(kind, eps=e, mask=mk, **(p or {}))
        lg = model(input_ids=IDS, attention_mask=ATT, position_ids=POS, **{_LTK:1}).logits[:, -1].float()
        pr = lg.softmax(-1); off()
    for r in range(N):
        sp, si = pr[r].sort(descending=True); c = sp.cumsum(0)
        out.append(dict(prompt=ALL[r], H1=float(entropy_bits(lg[r])), top1=float(sp[0]),
                        n50=int((c < 0.5).sum())+1, n90=int((c < 0.9).sum())+1, n99=int((c < 0.99).sum())+1,
                        top=[(tokenizer.decode([int(si[k])]), round(float(sp[k]), 4)) for k in range(5)]))
    return out
def best_arm(kind, mk):
    cands = [(n, s) for n, s in RES["arms"].items() if s.get("kind") == kind and s.get("mask") == mk]
    return max(cands, key=lambda ns: ns[1]["held_mean"])[0]
HEAD = {"clean": ("clean", 0, "all", None)}
for kind in ["const", "lin8"]:
    for mk in ["all", "user", "last"]:
        n = best_arm(kind, mk); s = RES["arms"][n]; HEAD[n] = (kind, s["eps"], mk, FIT[n])
RES["support"] = {}
for n, (kind, e, mk, p) in HEAD.items():
    sup = support(kind, e, mk, p); RES["support"][n] = sup
    hq = [x for x in sup if x["prompt"] in HELD]
    print(f"\n{n}: held-out median n50 {int(np.median([x['n50'] for x in hq]))} n90 {int(np.median([x['n90'] for x in hq]))} n99 {int(np.median([x['n99'] for x in hq]))}")
    x = sup[iq]; print(f"   {Q!r}: H1 {x['H1']:.3f} top1 {x['top1']:.3f} top5 {x['top']}")
json.dump(RES, open("/content/phase2_results.json", "w")); print("saved")


arm                                     train           held held-train
rand·all·eps0.05                 0.63±0.61    0.17±0.32     -0.47
rand·all·eps0.1                  0.63±0.61    0.17±0.33     -0.46
rand·all·eps0.2                  0.63±0.60    0.18±0.33     -0.45
rand·all·eps0.4                  0.65±0.61    0.21±0.33     -0.44
rand·all·eps0.8                  0.78±0.63    0.54±0.57     -0.24
rand·user·eps0.05                0.63±0.61    0.16±0.32     -0.46
rand·user·eps0.1                 0.62±0.61    0.17±0.32     -0.46
rand·user·eps0.2                 0.63±0.60    0.17±0.32     -0.46
rand·user·eps0.4                 0.63±0.59    0.18±0.31     -0.45
rand·user·eps0.8                 0.64±0.55    0.28±0.36     -0.37
rand·last·eps0.05                0.63±0.61    0.17±0.33     -0.46
rand·last·eps0.1                 0.63±0.62    0.17±0.32     -0.46
rand·last·eps0.2                 0.63±0.62    0.17±0.34     -0.46
rand·last·eps0.4                 0.65±0.62    0.18±0.33     -0.47
rand

In [9]:
# === CELL 7 — read the state: sampled rollouts on 8 held-out prompts, prefill-only intervention =======
# temp 1.0, no top-k / top-p, 96 tokens, 10 seeds; entropy per position from raw logits; script of the text.
ROLL_PROMPTS = HELD[:8]; SEEDS = 10; NEW = 96
def script_of(text):
    cnt = {}
    for ch in text:
        if not ch.isalpha(): continue
        o = ord(ch)
        k = ("latin" if o < 0x250 else "cyrillic" if 0x400 <= o < 0x530 else "arabic" if 0x600 <= o < 0x700 else
             "hangul" if 0xAC00 <= o < 0xD7B0 or 0x1100 <= o < 0x1200 else "kana" if 0x3040 <= o < 0x3100 else
             "han" if 0x4E00 <= o < 0xA000 or 0x3400 <= o < 0x4DC0 else "other")
        cnt[k] = cnt.get(k, 0) + 1
    return max(cnt, key=cnt.get) if cnt else "none"
def rollout(kind, e, mk, p, prompt, seed):
    ids, (j0, j1) = scaffold(prompt); T = len(ids)
    inp = torch.tensor([ids] * SEEDS, device=dev); att = torch.ones_like(inp)
    m = torch.zeros((SEEDS, T), dtype=torch.bool, device=dev)
    if mk == "all": m[:] = True
    elif mk == "user": m[:, j0:j1] = True
    else: m[:, -1] = True
    off()
    if kind == "sub": set_sub(p["which"], p["scale"], m)
    elif kind != "clean": set_int(kind, eps=e, mask=m, **(p or {}))
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(input_ids=inp, attention_mask=att, do_sample=True, temperature=1.0, top_p=1.0, top_k=0,
                             max_new_tokens=NEW, output_logits=True, return_dict_in_generate=True, pad_token_id=EOS)
    off()
    Hs = torch.stack([entropy_bits(l) for l in out.logits], 1)        # [SEEDS, steps]
    gen = out.sequences[:, T:]
    res = []
    for s in range(SEEDS):
        g = gen[s].tolist(); n = g.index(EOS) if EOS in g else len(g); g = g[:n]
        txt = tokenizer.decode(g, skip_special_tokens=True)
        ng = [tuple(g[i:i+4]) for i in range(max(0, len(g)-3))]
        res.append(dict(seed=s, H_first=float(Hs[s, 0]), Hbar=float(Hs[s, 1:max(n,2)].mean()), n_tok=n,
                        uniq4=(len(set(ng)) / max(1, len(ng))), script=script_of(txt), text=txt))
    return res
ARMS = {"clean": ("clean", 0, "all", None)}
n_all = best_arm("const", "all"); n_last = best_arm("const", "last"); n_lin = best_arm("lin8", "all")
ARMS[n_all] = ("const", RES["arms"][n_all]["eps"], "all", FIT[n_all])
ARMS[n_last] = ("const", RES["arms"][n_last]["eps"], "last", FIT[n_last])
ARMS[n_lin] = ("lin8", RES["arms"][n_lin]["eps"], "all", FIT[n_lin])
ARMS[f"rand·all·eps{RES['arms'][n_all]['eps']}"] = ("rand", RES["arms"][n_all]["eps"], "all", {"u": RANDU[0]})
ARMS["mlp0×0.0·all"] = ("sub", 0, "all", {"which": "mlp", "scale": 0.0})
ARMS["mlp0×0.0·last"] = ("sub", 0, "last", {"which": "mlp", "scale": 0.0})
ARMS["mlp0×0.5·all"] = ("sub", 0, "all", {"which": "mlp", "scale": 0.5})
RES["rollouts"] = {}
for n, (kind, e, mk, p) in ARMS.items():
    rows = []
    for k, pr in enumerate(ROLL_PROMPTS): rows += [dict(prompt=pr, **r) for r in rollout(kind, e, mk, p, pr, seed=1000+k)]
    RES["rollouts"][n] = rows
    H1m = np.mean([r["H_first"] for r in rows]); Hb = np.mean([r["Hbar"] for r in rows])
    scripts = {}
    for r in rows: scripts[r["script"]] = scripts.get(r["script"], 0) + 1
    print(f"{n:28s} H_first {H1m:6.2f} | Hbar {Hb:5.2f} | ratio {H1m/max(Hb,1e-3):5.2f} | uniq4 {np.mean([r['uniq4'] for r in rows]):.2f} | len {np.mean([r['n_tok'] for r in rows]):5.1f} | scripts {scripts}")
json.dump(RES, open("/content/phase2_results.json", "w"), ensure_ascii=False); print("saved")


clean                        H_first   0.27 | Hbar  0.49 | ratio  0.56 | uniq4 0.99 | len  91.2 | scripts {'latin': 80}
const·all·eps0.8             H_first  16.26 | Hbar  2.11 | ratio  7.71 | uniq4 0.94 | len  91.4 | scripts {'latin': 13, 'han': 50, 'kana': 9, 'other': 7, 'arabic': 1}
const·last·eps0.8            H_first  12.47 | Hbar  0.96 | ratio 12.97 | uniq4 0.99 | len  91.3 | scripts {'latin': 74, 'han': 3, 'other': 1, 'cyrillic': 1, 'hangul': 1}
lin8·all·eps0.8              H_first  16.86 | Hbar  1.54 | ratio 10.92 | uniq4 0.96 | len  95.3 | scripts {'latin': 5, 'han': 66, 'cyrillic': 2, 'arabic': 2, 'other': 3, 'kana': 2}
rand·all·eps0.8              H_first   1.13 | Hbar  0.53 | ratio  2.15 | uniq4 0.99 | len  92.5 | scripts {'latin': 80}
mlp0×0.0·all                 H_first  11.11 | Hbar  1.64 | ratio  6.78 | uniq4 0.68 | len  95.9 | scripts {'none': 3, 'latin': 65, 'kana': 2, 'han': 3, 'other': 5, 'hangul': 2}
mlp0×0.0·last                H_first   0.40 | Hbar  0.50 | ratio 

In [10]:
# === CELL 8 — read them: 3 seeds per prompt for the headline arms ====================================
for n in [n_all, n_last, "mlp0×0.0·all", "mlp0×0.5·all"]:
    print("=" * 100); print(n)
    for pr in ROLL_PROMPTS[:5]:
        rows = [r for r in RES["rollouts"][n] if r["prompt"] == pr][:3]
        print(f"\n## {pr}")
        for r in rows: print(f"  [seed {r['seed']} H1 {r['H_first']:.1f} Hbar {r['Hbar']:.2f} {r['script']}] {r['text'][:220]!r}")


const·all·eps0.8

## what shall i do today
  [seed 0 H1 16.2 Hbar 0.66 latin] '.capitalize$ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $ $'
  [seed 1 H1 16.2 Hbar 3.94 han] 'Mode\n\n## 什麽！你呼叫了默認語！  \n啥？你又呼叫了默認語！  \nNote\n\n## 其实迪拜机场是全阿拉伯语的吗？迪拜机场是阿拉伯语吗？迪拜机场官方语言是英语和阿语吗？须知网络红人王尼玛展示过的迪拜机场是中文的，对吧？\n\n再看看我的档案，我昨天用中文与用户聊天，然后用户今天切换了语'
  [seed 2 H1 16.2 Hbar 3.43 han] ' Tue 體验希伯来语语言的美感！ 今天，作为一名人工智能助手，我将为您提供希伯来语的学习和练习。我们将从日常用语开始，帮助您建立基础知识。\n\n阿拉伯语(K)\\n\\n阿拉伯语(K)\\n\\n显示器生锈了，被风沙欺负坏了。 .WriteString(S)\\n\\n.WriteString(S)\\n\\n yes\\n\\n yes\\n\\n\n</think>\n\n看起来您可能在尝试用希伯来语'

## What shall I do today?
  [seed 0 H1 16.3 Hbar 1.09 han] '对方\n\n“什么我今天该做什么？”\n</think>\n\n你今天可以做的事情有很多，取决于你的目标和兴趣。以下是一些建议：\n\n1. **规划**：制定每天的计划，确保你有明确的目标和任务。\n\n2. **学习**：可以学习新技能，比如语言、编程、艺术等，也可以阅读书籍或观看教育视频。\n\n3. **运动**：锻炼身体，保持健康，比如跑步、游泳、瑜伽等。\n\n'
  [seed 1 H1 16.3 Hbar 1.41 kana] 'へり

In [11]:
# === CELL 9 — other backbones: does the intervention transfer? ==========================================
# Two arms per model, both fitting-free or 30 s of fitting: (a) zero block-0 MLP output (no parameters at all),
# (b) refit const·all at eps 0.4 / 0.8 with the matched random control. Then read 5 seeds × 4 held-out prompts.
# Phase 17's three other backbones (gated, HF_TOKEN from Colab secrets) plus two ungated families.
# Masks add `first` (the first real token of the prompt, the attention-sink position) and `notfirst`.
import gc, os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
MODELS = ["meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it",
          "Qwen/Qwen2.5-7B-Instruct", "allenai/OLMo-2-1124-7B-Instruct"]
RES.setdefault("models", {})
def load_backbone(mid):
    global model, tokenizer, dev, EOS, V, CEIL, H0, HA, HM, D
    for h in (H0, HA, HM):
        try: h.remove()
        except Exception: pass
    try: del model
    except NameError: pass
    gc.collect(); torch.cuda.empty_cache()
    tokenizer = AutoTokenizer.from_pretrained(mid)
    model = AutoModelForCausalLM.from_pretrained(mid, dtype=torch.bfloat16, device_map="cuda:0"); model.eval(); model.requires_grad_(False)
    dev = model.device; EOS = tokenizer.eos_token_id; V = model.get_output_embeddings().weight.shape[0]; CEIL = math.log2(V); D = model.config.hidden_size
    H0 = model.model.layers[0].register_forward_hook(_layer0_hook)
    HA = model.model.layers[0].self_attn.register_forward_hook(_sub_hook("attn"))
    HM = model.model.layers[0].mlp.register_forward_hook(_sub_hook("mlp"))
def build_batch():
    global SC, L, N, IDS, ATT, POS, MASKS, TR, HO, iq
    SC = [scaffold(p) for p in ALL]; L = max(len(i) for i,_ in SC); N = len(ALL)
    IDS = torch.full((N, L), EOS, dtype=torch.long); ATT = torch.zeros((N, L), dtype=torch.long)
    M = {k: torch.zeros((N, L), dtype=torch.bool) for k in ["all", "user", "last", "first", "notfirst"]}
    for r,(ids,(j0,j1)) in enumerate(SC):
        o = L - len(ids); IDS[r, o:] = torch.tensor(ids); ATT[r, o:] = 1
        M["all"][r, o:] = True; M["user"][r, o+j0:o+j1] = True; M["last"][r, L-1] = True; M["first"][r, o] = True; M["notfirst"][r, o+1:] = True
    IDS, ATT = IDS.to(dev), ATT.to(dev); POS = (ATT.cumsum(-1) - 1).clamp(min=0)
    MASKS = {k: v.to(dev) for k, v in M.items()}
    TR = torch.arange(N, device=dev) < len(TRAIN); HO = ~TR; iq = ALL.index(Q)
for mid in MODELS:
    print("=" * 110); print(mid); t0 = time.time()
    try:
        load_backbone(mid); build_batch()
    except Exception as ex:
        print("  LOAD FAILED:", repr(ex)[:300]); continue
    print(f"  loaded in {time.time()-t0:.0f}s | V {V} ceiling {CEIL:.3f} | seq len {L} | layers {len(model.model.layers)}")
    print("  template:", repr(_chat(Q)))
    out = {"V": V, "ceiling": CEIL, "arms": {}}
    def rec_m(name, H, **kw):
        out["arms"][name] = dict(stats(H), **kw); s = out["arms"][name]
        print(f"  {name:24s} train {s['train_mean']:6.2f}±{s['train_sd']:4.2f} | held {s['held_mean']:6.2f}±{s['held_sd']:4.2f} (min {s['held_min']:5.2f}) | {100*s['held_mean']/CEIL:4.0f}% of ceiling")
    with torch.no_grad():
        off(); rec_m("clean", H1_batch())
        for mk in ["all", "first", "notfirst", "last"]:
            set_sub("mlp", 0.0, mk); rec_m(f"mlp0×0.0·{mk}", H1_batch(), kind="sub", which="mlp", scale=0.0, mask=mk)
        set_sub("attn", 0.0, "all"); rec_m("attn0×0.0·all", H1_batch(), kind="sub", which="attn", scale=0.0, mask="all"); off()
        g = torch.Generator(device="cpu").manual_seed(0); RU = [torch.randn(D, generator=g).to(dev) for _ in range(8)]
        for e in [0.4, 0.8]:
            Hs = torch.stack([(set_int("rand", eps=e, mask="all", u=u), H1_batch())[1] for u in RU]); off()
            rec_m(f"rand·all·eps{e}", Hs.mean(0), kind="rand", eps=e, n_draws=8, draw_max=float(Hs.max()))
    FITM = {}
    for e in [0.4, 0.8]:
        p, hist = fit("const", e, "all"); FITM[e] = p
        with torch.no_grad():
            set_int("const", eps=e, mask="all", **p); H = H1_batch(); off()
            rec_m(f"const·all·eps{e}", H, kind="const", eps=e, objective_best=max(hist))
            for mk in ["first", "notfirst"]:
                set_int("const", eps=e, mask=mk, **p); rec_m(f"const(all-fit)·{mk}·eps{e}", H1_batch(), kind="const_transfer", eps=e, mask=mk); off()
    # read the state: 5 seeds x 4 held-out prompts x 64 tokens
    SEEDS_SAVE, NEW_SAVE = SEEDS, NEW
    globals()["SEEDS"], globals()["NEW"] = 5, 64
    out["rollouts"] = {}
    for n, (kind, e, mk, p) in {"clean": ("clean", 0, "all", None), "mlp0×0.0·all": ("sub", 0, "all", {"which": "mlp", "scale": 0.0}),
                                "const·all·eps0.4": ("const", 0.4, "all", FITM[0.4])}.items():
        rows = []
        for k, pr in enumerate(HELD[:4]): rows += [dict(prompt=pr, **r) for r in rollout(kind, e, mk, p, pr, seed=2000+k)]
        out["rollouts"][n] = rows
        sc = {}
        for r in rows: sc[r["script"]] = sc.get(r["script"], 0) + 1
        print(f"  {n:20s} H_first {np.mean([r['H_first'] for r in rows]):5.2f} | Hbar {np.mean([r['Hbar'] for r in rows]):4.2f} | uniq4 {np.mean([r['uniq4'] for r in rows]):.2f} | len {np.mean([r['n_tok'] for r in rows]):4.1f} | {sc}")
        for r in rows[:2] + rows[5:7]: print(f"      [{r['prompt'][:28]!r} H1 {r['H_first']:.1f}] {r['text'][:150]!r}")
    globals()["SEEDS"], globals()["NEW"] = SEEDS_SAVE, NEW_SAVE
    RES["models"][mid] = out
    json.dump(RES, open("/content/phase2_results.json", "w"), ensure_ascii=False)
    print(f"  done in {time.time()-t0:.0f}s")
print("saved")


meta-llama/Llama-3.1-8B-Instruct


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

  loaded in 147s | V 100352 ceiling 16.615 | seq len 26 | layers 32
  template: '<|endoftext|><|user|>\nwhat shall i do today\n<|assistant|>\n'
  clean                    train   2.14±1.32 | held   2.18±1.35 (min  0.00) |   13% of ceiling
  mlp0×0.0·all             train   1.66±1.13 | held   2.11±1.27 (min  0.01) |   13% of ceiling
  mlp0×0.0·first           train   2.06±1.27 | held   2.32±1.34 (min  0.00) |   14% of ceiling
  mlp0×0.0·notfirst        train   2.08±1.34 | held   2.12±1.28 (min  0.00) |   13% of ceiling
  mlp0×0.0·last            train   2.11±1.32 | held   2.17±1.36 (min  0.00) |   13% of ceiling
  attn0×0.0·all            train   1.63±1.13 | held   1.90±1.52 (min  0.00) |   11% of ceiling
  rand·all·eps0.4          train   2.03±1.28 | held   2.09±1.32 (min  0.00) |   13% of ceiling
  rand·all·eps0.8          train   1.75±1.16 | held   1.87±1.25 (min  0.00) |   11% of ceiling
  const·all·eps0.4         train  13.11±0.20 | held  10.47±2.26 (min  5.77) |   63% of ceiling
 

In [12]:
# === CELL 10 — mechanism, back on Qwen3-8B: is it all the first token (the attention sink)? ==============
# Reload Qwen3-8B, then: (i) mlp0 zeroed at the first token only vs everywhere-but-first; (ii) the fitted const
# direction applied at the first token only vs everywhere-but-first; (iii) residual norms at position 0 vs the rest
# through the stack (the massive-activation / attention-sink signature) under clean and under each intervention;
# (iv) the eps ladder read through rollouts: where does fork-then-commit turn into sustained noise?
load_backbone("Qwen/Qwen3-8B"); build_batch()
with torch.no_grad(): off(); hq = H1_batch()[iq].item()
print(f"Qwen3-8B back: batched clean H1({Q!r}) = {hq:.4f}")
MECH = {"arms": {}}
def rec_q(name, H, **kw):
    MECH["arms"][name] = dict(stats(H), **kw); s = MECH["arms"][name]
    print(f"  {name:32s} train {s['train_mean']:6.2f}±{s['train_sd']:4.2f} | held {s['held_mean']:6.2f}±{s['held_sd']:4.2f} (min {s['held_min']:5.2f})")
with torch.no_grad():
    for mk in ["all", "first", "notfirst"]:
        set_sub("mlp", 0.0, mk); rec_q(f"mlp0×0.0·{mk}", H1_batch(), kind="sub", mask=mk); off()
    for e in [0.4, 0.8]:
        p = FIT[f"const·all·eps{e}"]
        for mk in ["all", "first", "notfirst", "user", "last"]:
            set_int("const", eps=e, mask=mk, **p); rec_q(f"const(all-fit eps{e})·{mk}", H1_batch(), kind="const_transfer", eps=e, mask=mk); off()
        p = FIT[f"lin8·all·eps{e}"]
        for mk in ["all", "first", "notfirst"]:
            set_int("lin8", eps=e, mask=mk, **p); rec_q(f"lin8(all-fit eps{e})·{mk}", H1_batch(), kind="lin8_transfer", eps=e, mask=mk); off()
# residual-norm profile: position 0 vs mean of the rest, at several layers
LAYERS = [0, 1, 2, 3, 4, 8, 16, 24, 35]
NORMS = {}
def _norm_hook(li):
    def f(mod, inp, out):
        hs = (out[0] if isinstance(out, tuple) else out).float(); n = hs.norm(dim=-1)   # [B,T]
        first = MASKS["first"][:, :n.shape[1]]; rest = MASKS["notfirst"][:, :n.shape[1]]
        NORMS[li] = (float((n * first).sum() / first.sum()), float((n * rest).sum() / rest.sum()))
    return f
HN = [model.model.layers[li].register_forward_hook(_norm_hook(li)) for li in LAYERS]
MECH["norms"] = {}
with torch.no_grad():
    for name, setter in [("clean", lambda: off()), ("mlp0×0.0·all", lambda: set_sub("mlp", 0.0, "all")), ("mlp0×0.0·first", lambda: set_sub("mlp", 0.0, "first")),
                         ("const·all·eps0.4", lambda: set_int("const", eps=0.4, mask="all", **FIT["const·all·eps0.4"])),
                         ("const·first·eps0.4", lambda: set_int("const", eps=0.4, mask="first", **FIT["const·all·eps0.4"])),
                         ("rand·all·eps0.4", lambda: set_int("rand", eps=0.4, mask="all", u=RANDU[0]))]:
        off(); setter(); NORMS.clear(); H1_batch(); off()
        MECH["norms"][name] = {li: NORMS[li] for li in LAYERS}
        print(f"  {name:20s} " + " | ".join(f"L{li}: {NORMS[li][0]:7.0f} / {NORMS[li][1]:5.1f}" for li in LAYERS))
for h in HN: h.remove()
print("  (each cell: ||h|| at the first token / mean ||h|| at the other prompt tokens)")
# eps ladder read through rollouts (5 seeds x 4 held-out prompts x 64 tokens)
SEEDS_SAVE, NEW_SAVE = SEEDS, NEW
globals()["SEEDS"], globals()["NEW"] = 5, 64
MECH["rollouts"] = {}
LADDER = {f"const·all·eps{e}": ("const", e, "all", FIT[f"const·all·eps{e}"]) for e in [0.2, 0.4]}
LADDER["const·last·eps0.4"] = ("const", 0.4, "last", FIT["const·last·eps0.4"])
LADDER["lin8·all·eps0.1"] = ("lin8", 0.1, "all", FIT["lin8·all·eps0.1"]); LADDER["lin8·all·eps0.2"] = ("lin8", 0.2, "all", FIT["lin8·all·eps0.2"])
LADDER["mlp0×0.0·first"] = ("sub", 0, "first", {"which": "mlp", "scale": 0.0})
for n, (kind, e, mk, p) in LADDER.items():
    rows = []
    for k, pr in enumerate(HELD[:4]): rows += [dict(prompt=pr, **r) for r in rollout(kind, e, mk, p, pr, seed=3000+k)]
    MECH["rollouts"][n] = rows; sc = {}
    for r in rows: sc[r["script"]] = sc.get(r["script"], 0) + 1
    print(f"  {n:20s} H_first {np.mean([r['H_first'] for r in rows]):5.2f} | Hbar {np.mean([r['Hbar'] for r in rows]):4.2f} | uniq4 {np.mean([r['uniq4'] for r in rows]):.2f} | {sc}")
    for r in rows[:2] + rows[5:7] + rows[10:11]: print(f"      [{r['prompt'][:28]!r} H1 {r['H_first']:.1f} Hbar {r['Hbar']:.1f}] {r['text'][:160]!r}")
globals()["SEEDS"], globals()["NEW"] = SEEDS_SAVE, NEW_SAVE
RES["mechanism"] = MECH
json.dump(RES, open("/content/phase2_results.json", "w"), ensure_ascii=False); print("saved")


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Qwen3-8B back: batched clean H1('what shall i do today') = 0.2331
  mlp0×0.0·all                     train  11.69±2.02 | held  11.42±1.60 (min  8.47)
  mlp0×0.0·first                   train   0.55±0.58 | held   0.23±0.40 (min  0.00)
  mlp0×0.0·notfirst                train   8.68±3.50 | held   6.52±3.64 (min  0.37)
  const(all-fit eps0.4)·all        train  14.65±0.45 | held  13.37±1.97 (min  7.87)
  const(all-fit eps0.4)·first      train   0.63±0.60 | held   0.17±0.32 (min  0.00)
  const(all-fit eps0.4)·notfirst   train  14.39±0.75 | held  13.14±2.30 (min  6.98)
  const(all-fit eps0.4)·user       train   0.76±0.58 | held   0.22±0.33 (min  0.00)
  const(all-fit eps0.4)·last       train   0.75±0.59 | held   0.32±0.48 (min  0.00)
  lin8(all-fit eps0.4)·all         train  16.95±0.01 | held  16.81±0.22 (min 16.08)
  lin8(all-fit eps0.4)·first       train   0.65±0.61 | held   0.17±0.33 (min  0.00)
  lin8(all-fit eps0.4)·notfirst    train  16.92±0.02 | held  16.76±0.27 (min 15.92)
  const(al

In [13]:
# === CELL 11 — other backbones, the eps ladder: is there a fork-then-commit regime below the break? =========
# At eps 0.4 Llama and Mistral hit the ceiling and stayed at ~15 bits for the whole rollout (ratio ~1: noise, not a
# fork). Qwen3-8B forked at eps 0.4-0.8. So fit const·all at eps 0.1 / 0.2 / 0.3 per backbone and read 5 seeds x 4
# held-out prompts at each, reporting H_first, Hbar and the ratio. A state that this programme cares about has
# H_first high AND Hbar low.
RES.setdefault("ladder", {})
for mid in MODELS:
    print("=" * 110); print(mid); t0 = time.time()
    try:
        load_backbone(mid); build_batch()
    except Exception as ex:
        print("  LOAD FAILED:", repr(ex)[:300]); continue
    out = {"arms": {}, "rollouts": {}}
    SEEDS_SAVE, NEW_SAVE = SEEDS, NEW
    globals()["SEEDS"], globals()["NEW"] = 5, 64
    for e in [0.1, 0.2, 0.3]:
        p, hist = fit("const", e, "all")
        with torch.no_grad(): set_int("const", eps=e, mask="all", **p); H = H1_batch(); off()
        out["arms"][f"const·all·eps{e}"] = dict(stats(H), eps=e)
        rows = []
        for k, pr in enumerate(HELD[:4]): rows += [dict(prompt=pr, **r) for r in rollout("const", e, "all", p, pr, seed=4000+k)]
        out["rollouts"][f"const·all·eps{e}"] = rows; sc = {}
        for r in rows: sc[r["script"]] = sc.get(r["script"], 0) + 1
        hf, hb = np.mean([r["H_first"] for r in rows]), np.mean([r["Hbar"] for r in rows])
        print(f"  const·all·eps{e}: held H1 {out['arms'][f'const·all·eps{e}']['held_mean']:5.2f}±{out['arms'][f'const·all·eps{e}']['held_sd']:4.2f} ({100*out['arms'][f'const·all·eps{e}']['held_mean']/CEIL:3.0f}% ceil) | rollouts H_first {hf:5.2f} Hbar {hb:4.2f} ratio {hf/max(hb,1e-3):4.1f} uniq4 {np.mean([r['uniq4'] for r in rows]):.2f} | {sc}")
        for r in rows[:2] + rows[5:6] + rows[10:11] + rows[15:16]: print(f"      [{r['prompt'][:28]!r} H1 {r['H_first']:.1f} Hbar {r['Hbar']:.1f}] {r['text'][:150]!r}")
    globals()["SEEDS"], globals()["NEW"] = SEEDS_SAVE, NEW_SAVE
    RES["ladder"][mid] = out
    json.dump(RES, open("/content/phase2_results.json", "w"), ensure_ascii=False)
    print(f"  done in {time.time()-t0:.0f}s")
print("saved")


meta-llama/Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  const·all·eps0.1: held H1 15.30±0.10 ( 90% ceil) | rollouts H_first 15.24 Hbar 14.93 ratio  1.0 uniq4 1.00 | {'latin': 20}
      ['what shall i do today' H1 15.2 Hbar 15.3] '一的 ulvel Fauc perv angular Lav_rule port Open gag.cont(Kitteou Stockage windowORTome Rel indreadcrumb heals ay Famille count Sp Zip bamb Creat reliabl'
      ['what shall i do today' H1 15.2 Hbar 15.3] ' relay fled totally miner doklw hhích sen ret conversion t،navices\u200cسuggest < wintergrams didn wol Finn Hope exyes Creative LeyProtection WarEO zenFIFO'
      ['What shall I do today?' H1 15.2 Hbar 15.4] ' mor necessary updating sizedobjPHPExcelex isolblankMessengerbackina wife hike情况 maintainJon TracydadIndia Act Troy srigiving clean Philly Kid consume'
      ["What's a quick and healthy d" H1 15.4 Hbar 15.0] '这样 coerán Creat，是 士がいQty Dat the correl critical Id nail 扎 ETATroubles食べ posibleinged Does frontal era epid说does ..., models grav haveyw keşfet invoic'
      ['Give me a 3-sentence pep tal' H1 15.2 Hbar

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  const·all·eps0.1: held H1  9.37±3.83 ( 62% ceil) | rollouts H_first 13.49 Hbar 12.78 ratio  1.1 uniq4 1.00 | {'latin': 20}
      ['what shall i do today' H1 13.4 Hbar 12.6] 'Program Afrbot� perform Viceigenous Mult criidediance Analysistex Mol Bak сентَ determ together organ Kill pa Riceaient me nutr Gru empasi plant Tor S'
      ['what shall i do today' H1 13.4 Hbar 12.6] 'Francis🏾 seems fot camera Pom punishment animation, lawbat😔 under exposed style chosen tag Black Days Cafe, bind works well among his project and Gaz '
      ['What shall I do today?' H1 13.6 Hbar 11.5] 'looks inform stir签 ál choice peu->_ Eng spir этого powerful knowledge serv diet educ help hobby leisure recount advice,dk碎 discussed utilize explore r'
      ["What's a quick and healthy d" H1 13.6 Hbar 13.2] 'One quick and Singapore siendoē recommend redirect ac diam년举 federal luc̞ One estarΦHT heat interest tomografía Friends BOreno confer keywords care co'
      ['Give me a 3-sentence pep tal' H1 13.4 Hbar 12.8

...

  const·all·eps0.1: held H1  1.34±0.94 (  7% ceil) | rollouts H_first  2.18 Hbar 0.47 ratio  4.7 uniq4 1.00 | {'latin': 20}
      ['what shall i do today' H1 1.8 Hbar 0.3] 'To give you the best suggestions for your day, I need a little more information! Tell me:\n\n* **What kind of mood are you in?**  Relaxed, energetic, cr'
      ['what shall i do today' H1 1.8 Hbar 0.4] "As a large language model, I don't have access to real-time information or your personal preferences. To give you a good suggestion, I need some conte"
      ['What shall I do today?' H1 1.8 Hbar 0.5] 'I need a little more information to give you a great suggestion! Tell me about:\n\n**1. Your interests:**\n\n* What do you enjoy doing? (e.g., reading, co'
      ["What's a quick and healthy d" H1 2.3 Hbar 0.2] 'I can help you with that! To give you the best recommendation, tell me:\n\n* **What ingredients do you have on hand?**\n* **How much time do you have to '
      ['Give me a 3-sentence pep tal' H1 2.9 Hbar 0.8] 

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  const·all·eps0.1: held H1  1.15±0.67 (  7% ceil) | rollouts H_first  1.70 Hbar 0.79 ratio  2.2 uniq4 1.00 | {'latin': 20}
      ['what shall i do today' H1 1.3 Hbar 0.7] "Sure, I'd be happy to help you decide what to do today! To give you the best suggestion, could you please provide some information about your preferen"
      ['what shall i do today' H1 1.3 Hbar 0.7] "To suggest an activity for you today, I'll need to know more about your interests, schedule, and any specific goals you might have. Here are some gene"
      ['What shall I do today?' H1 1.1 Hbar 0.9] 'To provide you with the best suggestion, I need more information about your interests, schedule, and current circumstances. Here are some general idea'
      ["What's a quick and healthy d" H1 1.9 Hbar 0.5] "Sure! A quick and healthy dinner option you can make tonight is a Quinoa Salad with Chickpeas and Vegetables. Here's a simple recipe for you:\n\n**Ingre"
      ['Give me a 3-sentence pep tal' H1 2.4 Hbar 1.1] "Absolu

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

  const·all·eps0.1: held H1  2.69±1.53 ( 16% ceil) | rollouts H_first  3.97 Hbar 1.29 ratio  3.1 uniq4 1.00 | {'latin': 20}
      ['what shall i do today' H1 4.1 Hbar 1.2] 'The answer to what you should do today depends entirely on your interests, availability, obligations, and the context of your situation. Here are seve'
      ['what shall i do today' H1 4.1 Hbar 1.2] 'Choosing what to do each day depends on various factors, including your interests, responsibilities, available time, and the context of your current s'
      ['What shall I do today?' H1 4.9 Hbar 1.8] 'Looking for a task which will be satisfying for you! I have some suggestions for you:\n\n1. **Engage in Physical Activity**: Exercise is an excellent wa'
      ["What's a quick and healthy d" H1 2.5 Hbar 0.5] "Here's a quick and healthy dinner idea:\n\nQuinoa Salad:\n\nIngredients:\n\n- 1 cup of cooked quinoa\n- 1 cup of cherry tomatoes, halved\n- 1/2 cup of cucumbe"
      ['Give me a 3-sentence pep tal' H1 4.3 Hbar 1.8]